In [ ]:
# Install hypertools (dev-1.0 preview) -- run this first on Colab.
# On release this becomes: %pip install hypertools
%pip install -q "hypertools[interactive] @ git+https://github.com/ContextLab/hypertools.git@dev-1.0"
%pip install -q sentence-transformers

%matplotlib inline

# The shape of a conversation

This tutorial turns a conversation into geometry. Each **turn** (a contiguous run of speech by one speaker) is embedded as a little sliding-window trajectory; all windows share one reduced 3-D space. Each turn is its own *disjoint* trajectory, colored by **speaker**, and `animate='serial'` reveals them one turn at a time.

We use Lewis Carroll's *Mad Tea-Party*, quoted verbatim from the [Project Gutenberg](https://www.gutenberg.org) text and bundled below so the notebook is self-contained.

**What gets embedded is spoken text only.** Every narrative attribution ("said the Hatter", "Alice replied") and all surrounding narration has been stripped, so the geometry reflects what the characters *say*, not how the narrator introduces them -- otherwise every one of Alice's turns would be pulled together by the repeated words "said Alice" rather than by their content. For the same reason the caption shows only the quoted line: who is speaking is carried by the colour, the legend and a speaker label under the title.

Turns are embedded with a sentence-transformer when available, falling back to a character n-gram TF-IDF vector otherwise.

## 1. Imports and the bundled turns

In [2]:
import numpy as np
import hypertools as hyp

SPEAKER_COLOR = {
    'Alice': '#E4572E', 'Hatter': '#3F72AF',
    'March Hare': '#5B8C5A', 'Dormouse': '#B5537F',
}

# CURATED, SPOKEN TEXT ONLY: each entry is exactly what that character says
# out loud, quoted verbatim from the Gutenberg text (emphasis underscores
# dropped), with every narrative attribution and all narration removed.
# Automatic extraction was tried and rejected: it mis-merged speakers across
# adjacent quotes and dragged narration into the embedded text.
TURNS = [
    ('Alice', "There's plenty of room!"),
    ('March Hare', "Have some wine."),
    ('Alice', "I don't see any wine."),
    ('March Hare', "There isn't any."),
    ('Alice', "Then it wasn't very civil of you to offer it."),
    ('March Hare', "It wasn't very civil of you to sit down without being invited."),
    ('Alice', "I didn't know it was your table; it's laid for a great many more than three."),
    ('Hatter', "Your hair wants cutting."),
    ('Alice', "You should learn not to make personal remarks; it's very rude."),
    ('Hatter', "Why is a raven like a writing-desk?"),
    ('Alice', "I'm glad they've begun asking riddles. I believe I can guess that."),
    ('March Hare', "Do you mean that you think you can find out the answer to it?"),
    ('Alice', "Exactly so."),
    ('March Hare', "Then you should say what you mean."),
    ('Alice', "I do; at least I mean what I say - that's the same thing, you know."),
    ('Hatter', "Not the same thing a bit! You might just as well say that 'I see what I eat' is the same thing as 'I eat what I see'!"),
    ('March Hare', "You might just as well say that 'I like what I get' is the same thing as 'I get what I like'!"),
    ('Dormouse', "You might just as well say that 'I breathe when I sleep' is the same thing as 'I sleep when I breathe'!"),
    ('Hatter', "It is the same thing with you. What day of the month is it?"),
    ('Alice', "The fourth."),
    ('Hatter', "Two days wrong! I told you butter wouldn't suit the works!"),
    ('March Hare', "It was the best butter."),
    ('Hatter', "Yes, but some crumbs must have got in as well; you shouldn't have put it in with the bread-knife."),
    ('Alice', "What a funny watch! It tells the day of the month, and doesn't tell what o'clock it is!"),
    ('Hatter', "Why should it? Does your watch tell you what year it is?"),
    ('Alice', "Of course not; but that's because it stays the same year for such a long time together."),
    ('Hatter', "Which is just the case with mine."),
    ('Alice', "I don't quite understand you."),
]

## 2. Embed each turn's sliding windows

`embed` prefers a sentence-transformer and falls back to a TF-IDF vector, so the notebook runs without a large model download. `word_spans` returns each window as a `(start, end)` pair of word indices rather than a joined string, so the caption can later bold the words of the window being drawn; the embedded text is still `' '.join(words[start:end])`.

In [3]:
def embed(texts):
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('all-MiniLM-L6-v2')
        return np.asarray(model.encode(texts, show_progress_bar=False),
                          dtype=float)
    except Exception:
        from sklearn.feature_extraction.text import TfidfVectorizer
        vec = TfidfVectorizer(analyzer='char_wb',
                              ngram_range=(2, 4), min_df=1)
        return vec.fit_transform(texts).toarray().astype(float)


def word_spans(text, size=6, step=2, min_wins=3):
    """Sliding word windows over one turn, as `(start, end)` index pairs.

    `min_wins` prevents a real rendering artifact. `hyp.plot` draws a ONE-ROW
    dataset as a dot (marker='.', linestyle='None'), because there is no line
    through a single point. With a fixed 6-word window, every turn of six words
    or fewer ("Have some wine.", "Exactly so.", "The fourth.") collapses to a
    single window and shows up as a stray dot floating in the box. Shrinking
    the window, and the step if needed, keeps every turn a real path.
    """
    w = text.split()
    n = len(w)
    size = max(1, min(size, n - min_wins + 1))
    step = step if (n - size) // step + 1 >= min_wins else 1
    return [(i, i + size) for i in range(0, n - size + 1, step)]

turn_words = [text.split() for _spk, text in TURNS]
turn_spans = [word_spans(text) for _spk, text in TURNS]
n_wins = [len(spans) for spans in turn_spans]
flat = [' '.join(words[a:b])
        for words, spans in zip(turn_words, turn_spans) for a, b in spans]
vecs = embed(flat)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 3. Reduce to a shared 3-D space and split into per-turn paths

All windows are reduced together (UMAP) so the turns live in the same space, then we split them back into one **disjoint** trajectory per turn, each with its speaker's color.

The UMAP kwargs are chosen for short turns. **`n_neighbors=8`** keeps the neighbor graph local: a turn contributes only a handful of windows, so a wide neighborhood would blur them into the global average instead of preserving each turn's own shape. **`min_dist=0.5`** spreads the embedded points out, so one turn's windows stay a readable path rather than collapsing into a blob on top of the next turn's. **`random_state=1`** fixes UMAP's stochastic optimization, so the embedding is reproducible from run to run.

In [4]:
import warnings
warnings.filterwarnings('ignore',
                        message=r'n_jobs value \d+ overridden.*',
                        category=UserWarning)  # benign UMAP/random_state note

red = np.asarray(hyp.reduce(
    vecs, reduce={'model': 'UMAP',
                  'kwargs': {'n_neighbors': 8, 'min_dist': 0.5,
                             'random_state': 1}}, ndims=3))
# no rescaling here: hyp.plot already mean-centers every dataset and rescales
# them into [-1, 1] with ONE shared affine before drawing

trajectories, colors, speakers = [], [], []
k = 0
for (spk, _text), nw in zip(TURNS, n_wins):
    trajectories.append(red[k:k + nw])
    k += nw
    colors.append(SPEAKER_COLOR[spk])
    speakers.append(spk)

## 4. Plot with `animate='serial'` + a recency fade, colored by speaker

Passing a *list* of arrays makes each turn a disjoint path; `animate='serial'` reveals them one turn at a time, accumulating the whole conversation. On top of the library call we add a hand-built **recency fade**: the current turn is fully opaque and each earlier turn is progressively more transparent (down to a floor) so the whole shape stays visible while the newest line stands out. There are no chemtrails -- the fade *is* the trail.

A label under the title names the speaker of the moment in that speaker's colour, and the caption tracks the current line, showing **only the quoted words** with the words of the window being drawn right now in **bold**. A single matplotlib `Text` cannot mix weights on one line, so the caption is built from per-word `TextArea`s packed into `HPacker` rows inside a `VPacker`, rebuilt each frame.

`duration=12` and `frame_rate=16` set the clip's length in seconds and its frames per second, so the animation is `duration * frame_rate = 192` frames long. That product (`total` in the code below) is the frame index every custom per-frame hook is driven by, which is why both are passed explicitly instead of left at the defaults.

One subtlety: `hyp.plot` resamples every multi-row line dataset onto the frame grid, so the *drawn* per-turn row counts are not the original turn lengths. The serial reveal is paced by the drawn lengths, so the current turn -- and the window index the bolding needs, recovered from the drawn row count -- must be derived from those (`ani._args[0]`); using the original lengths makes the opaque highlight lag the turn actually being drawn.

In [5]:
duration, fps = 12, 16
fig, ani = hyp.plot(trajectories, fmt='-', color=colors,
                    linewidth=1.6, animate='serial',
                    duration=duration, frame_rate=fps, elev=16,
                    size=(7.6, 7.4), show=False)

import matplotlib.patches as mpatches
from matplotlib.offsetbox import (TextArea, HPacker, VPacker,
                                  AnchoredOffsetbox)

present = [s for s in SPEAKER_COLOR if s in speakers]
fig.legend(handles=[mpatches.Patch(color=SPEAKER_COLOR[s], label=s)
                    for s in present], loc='upper left',
           bbox_to_anchor=(0.02, 0.93), frameon=False, fontsize=10)
fig.text(0.5, 0.965, "Alice's Mad Tea-Party", ha='center',
         va='top', fontsize=16, fontweight='bold', color='#1a1a1a')
# who is speaking right now, in that speaker's colour, under the title
speaker = fig.text(0.5, 0.923, '', ha='center', va='top',
                   fontsize=13, fontweight='bold')

n_turns = len(trajectories)
total = int(round(fps * duration))
lines = ani._args[1]                       # one Line3D per turn
# the DRAWN row counts (not the original turn lengths) pace the serial reveal
drawn_lens = [np.asarray(a).shape[0] for a in ani._args[0]]
starts = np.cumsum([0] + drawn_lens[:-1])
total_pts = int(sum(drawn_lens))
FLOOR, DECAY = 0.10, 0.45                   # oldest-turn floor; per-turn fade
# Over the final stretch the whole conversation is lifted back up, so the clip
# ends on the shape it spent the whole run building rather than on one lit turn
# against near-invisible history.
FINALE = int(1.4 * fps)
FINALE_FLOOR = 0.62
_orig = ani._func


def shown_counts(num):
    """Per-turn drawn row counts at this frame, mirroring
    update_lines_serial: revealed = total_points * num / (total_frames - 1)."""
    revealed = total_pts * num / max(1, total - 1)
    return [int(np.clip(revealed - st, 0, n))
            for st, n in zip(starts, drawn_lens)]


def current_state(num):
    """The (turn, window) being revealed right now, mirroring
    `update_lines_serial`: `revealed = total_points * num / (total_frames - 1)`,
    and a turn is ACTIVE while `0 < its shown-count < its row count`."""
    revealed = total_pts * num / max(1, total - 1)
    done = -1
    for j, (s, n) in enumerate(zip(starts, drawn_lens)):
        # int(clip(...)) mirrors update_lines_serial EXACTLY -- comparing the
        # un-truncated float disagrees with the backend on boundary frames
        shown = int(np.clip(revealed - s, 0, n))
        if 0 < shown < n:
            # the drawn rows are a resampling of this turn's windows, so map
            # the drawn position back onto a window index
            frac = (shown - 1) / max(1, n - 1)
            return j, int(round(frac * (n_wins[j] - 1)))
        if shown >= n:
            done = j                        # fully revealed
    if done < 0:
        # nothing drawn yet (frame 0). Falling through to the "between turns"
        # branch here reported the LAST window of turn 0, so the very first
        # frame bolded the end of the line and frame 1 snapped back.
        return 0, 0
    return done, n_wins[done] - 1                    # between turns


def caption_lines(ti, wi, width=68):
    """The turn's words as [[(word, is_bold), ...], ...] -- one list per
    wrapped line -- with the words of the window being drawn RIGHT NOW bold."""
    words = list(turn_words[ti])
    a, b = turn_spans[ti][wi]
    words[0] = '“' + words[0]
    words[-1] = words[-1] + '”'
    rows, row, used = [], [], 0
    for k, word in enumerate(words):
        step = len(word) + (1 if row else 0)
        if row and used + step > width:
            rows.append(row)
            row, used, step = [], 0, len(word)
        row.append((word, a <= k < b))
        used += step
    if row:
        rows.append(row)
    return rows


# The caption mixes bold and regular runs on ONE line, which a single Text
# artist cannot do, so it is built from per-word TextAreas packed into rows
# and rebuilt every frame. `sep` is the width of a space at this font size
# (0.318 em in DejaVu Sans).
CAP_FS = 12
caption = [None]                            # current artist


def set_caption(rows, color):
    if caption[0] is not None:
        caption[0].remove()
    packed = [HPacker(children=[
        TextArea(word, textprops=dict(color=color, fontsize=CAP_FS,
                                      style='italic',
                                      fontweight='bold' if bold else 'normal'))
        for word, bold in row], align='baseline', pad=0, sep=0.318 * CAP_FS)
        for row in rows]
    box = AnchoredOffsetbox(loc='lower center', pad=0, frameon=False,
                            child=VPacker(children=packed, align='center',
                                          pad=0, sep=4),
                            bbox_to_anchor=(0.5, 0.05),
                            bbox_transform=fig.transFigure)
    fig.add_artist(box)
    caption[0] = box


def _wrapped(num, *args):
    result = _orig(num, *args)
    ti, wi = current_state(num)
    # recency fade: the current turn is opaque; earlier turns get progressively
    # more transparent (a fading tail) down to a floor so the whole shape stays
    # visible; not-yet-spoken turns are hidden.
    counts = shown_counts(num)
    ramp = min(1.0, max(0.0, (num - (total - 1 - FINALE)) / max(1, FINALE)))
    floor = FLOOR + (FINALE_FLOOR - FLOOR) * ramp
    for j, ln in enumerate(lines):
        if j > ti or counts[j] < 2:
            # not yet spoken, or only ONE point drawn so far: hyp.plot renders
            # a single drawn point as a lone dot, which flashes as a speck for
            # one frame at each turn boundary.
            ln.set_alpha(0.0)
        elif j == ti:
            ln.set_alpha(1.0)
        else:
            ln.set_alpha(floor + (1.0 - floor) * DECAY ** (ti - j))
    # who is speaking, then the SPOKEN LINE only -- no attribution is tacked
    # onto the quote -- with the window being drawn right now in bold
    spk = speakers[ti]
    speaker.set_text(spk)
    speaker.set_color(SPEAKER_COLOR[spk])
    set_caption(caption_lines(ti, wi), SPEAKER_COLOR[spk])
    return result


ani._func = _wrapped

## 5. Display the animation

In [6]:
fig.set_dpi(100)  # halve hypertools' default 200-dpi canvas for a lighter GIF
ani.save('conversation_shape.gif', fps=fps)
print('saved conversation_shape.gif')

saved conversation_shape.gif


![the shape of a conversation, animated](conversation_shape.gif)